# Bandcamp Artist Discog Scraper
Based on:
- nahnhh's [webcrawl2_movie details.ipynb](https://github.com/nahnhh/top-movies-2005-2025/blob/main/webcrawl2_movie%20details.ipynb)
- diprog's [tls-client-async](https://github.com/diprog/python-tls-client-async) for better client identifiers.
- greasyfork's [Bandcamp: Show more dates](https://greasyfork.org/en/scripts/537005-bandcamp-show-more-dates/code)
- dbeley's [bandcamp-library-scraper.py](https://github.com/dbeley/bandcamp-library-scraper/blob/main/bandcamp-library-scraper.py) for `extract-discography` function

Artist list used: `slushwave-bandcamp-links.txt` compiled in Slushwave Social Club Discord server.

In [1]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import logging
from rich.logging import RichHandler

FORMAT = '%(message)s'
logging.basicConfig(
    level="INFO", format=FORMAT, datefmt="[%X]", handlers=[RichHandler()]
)
log = logging.getLogger('rich')

### Use `tls-client` instead of plain requests
TLS fingerprinting gives a more browser-like behavior to bypass anti-bot protections.

In [2]:
from bs4 import BeautifulSoup
from async_tls_client import AsyncSession
import asyncio
import os
import random
import json

In [3]:
GOOD_PROFILES = Path("good_profiles.json")
with open(GOOD_PROFILES, "r") as f:
		OK_CLIENTS = json.load(f)
class BrowserSession:
	def __init__(self):
		self.requests_made = 0
		self.new_session()
		self.retire_after = random.randint(40, 100)

	def rotate_client(self):
		self.client_identifier = random.choice(OK_CLIENTS)

	def new_session(self):
		self.rotate_client()
		self.session = AsyncSession(
			client_identifier=self.client_identifier,
			random_tls_extension_order=True
		)

		self.session.proxies.update({
			"http": os.getenv("mobileproxyuk"),
			"https": os.getenv("mobileproxyuk"),
		})

		self.session.headers.update({
			"Referer": "https://bandcamp.com/",
			"Accept-Language": "en-US,en;q=0.9",
		})

	async def get(self, url, **kwargs):
		if self.requests_made >= self.retire_after:
			self.new_session()
			self.requests_made = 0
			self.retire_after = random.randint(40, 100)

		resp = await self.session.get(url,**kwargs)
		self.requests_made += 1
		return resp

In [4]:
async def fetch(urls):
	random.seed(42)
	s = BrowserSession()
	failed = []
	soups = []
	for url in urls:
		r = await s.get(url)
		soup = BeautifulSoup(r.text, 'lxml')
		if soup.title and soup.title.get_text(strip=True) == "Client Challenge":
			failed.append({
				"url": url,
				"profile": s.client_identifier,
				})
			log.warning(f"Client Challenge with {s.client_identifier} for {url}")
			s.new_session()
			continue
		soups.append(soup)
	return soups

In [5]:
import re
def nozero(text):
	return re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)

### Fetching soup from urls: music page

In [6]:
urls = [
	'https://illusionarydreaming.bandcamp.com/', # custom grid, no music-grid
	'https://itachitsukiyomi.bandcamp.com/', # 4 columns
	'https://giftsfromhome.bandcamp.com/', # 3 columns
	'https://blackmoon00.bandcamp.com/', # 2 columns
	]
	
soups = await fetch(urls)

print("Albums fetched:")
print([soup.title.get_text(strip=True) for soup in soups])

Albums fetched:
['Illusionary ドリーミング', 'Music | itachi ツクヨミ', 'Music | Quà từ Nhà', 'Music | blackmoonブラックムーン']


#### Extract album urls from music page

In [7]:
def extract_album_urls(music_page_soup, artist_url=None):
	"""Extract all album urls from the artist's Music page"""
	if not artist_url:
		artist_url = music_page_soup.select_one('meta[property="og:url"]').get('content')
	links = (
		music_page_soup.select("li.music-grid-item a[href]")
		or music_page_soup.select("div.ipCellImage a[href]")
	)
	albums = []
	for a in links:
		href = a["href"]
		if href.startswith("http"):
			albums.append(href)
		else:
			albums.append(artist_url.rstrip("/") + href)

	return albums

In [8]:
extract_album_urls(soups[3])[:3]

['https://blackmoon00.bandcamp.com/album/--4',
 'https://blackmoon00.bandcamp.com/album/--3',
 'https://blackmoon00.bandcamp.com/album/--2']

### Metadata of one track

In [9]:
t_url = 'https://noproblematapes.bandcamp.com/track/feat-memorykeeper7'
r = await BrowserSession().get(t_url)
soup = BeautifulSoup(r.text, 'lxml')
t_schema = json.loads(soup.select("script[type='application/ld+json']")[0].get_text(strip=True))
t_schema.keys()

dict_keys(['@type', '@id', 'additionalProperty', 'name', 'duration', 'dateModified', 'datePublished', 'inAlbum', 'byArtist', 'publisher', 'copyrightNotice', 'keywords', 'image', 'mainEntityOfPage', '@context'])

In [10]:
[
  t_schema['byArtist']['name'],
	t_schema['name'],
  t_schema['image'],
]

['NO PROBLEMA TAPES',
 'لا طريق آخر (feat. memorykeeper7)',
 'https://f4.bcbits.com/img/a2569838638_10.jpg']

In [11]:
tralbum_tag = soup.select_one("[data-tralbum]")
t_tralbum = json.loads(tralbum_tag.get("data-tralbum", "{}"))
t_tralbum.keys()

dict_keys(['for the curious', 'current', 'preorder_count', 'hasAudio', 'art_id', 'packages', 'defaultPrice', 'freeDownloadPage', 'FREE', 'PAID', 'artist', 'item_type', 'id', 'last_subscription_item', 'has_discounts', 'is_bonus', 'play_cap_data', 'is_purchased', 'items_purchased', 'is_private_stream', 'is_band_member', 'licensed_version_ids', 'package_associated_license_id', 'has_video', 'tralbum_subscriber_only', 'album_is_preorder', 'album_release_date', 'trackinfo', 'playing_from', 'album_url', 'album_upsell_url', 'url'])

In [12]:
t_tralbum.get('art_id')

2569838638

In [13]:
t_tralbum['trackinfo'][0].keys()

dict_keys(['id', 'track_id', 'file', 'artist', 'title', 'encodings_id', 'license_type', 'private', 'track_num', 'album_preorder', 'unreleased_track', 'title_link', 'has_lyrics', 'has_info', 'streaming', 'is_downloadable', 'has_free_download', 'free_album_download', 'duration', 'lyrics', 'sizeof_lyrics', 'is_draft', 'video_source_type', 'video_source_id', 'video_mobile_url', 'video_poster_url', 'video_id', 'video_caption', 'video_featured', 'alt_link', 'encoding_error', 'encoding_pending', 'play_count', 'is_capped', 'track_license_id'])

In [14]:
t_tralbum['trackinfo'][0]['track_num']

12

In [15]:
t_tralbum.get('current').keys()

dict_keys(['audit', 'title', 'new_date', 'mod_date', 'publish_date', 'private', 'killed', 'download_pref', 'require_email', 'is_set_price', 'set_price', 'minimum_price', 'minimum_price_nonzero', 'require_email_0', 'artist', 'about', 'credits', 'auto_repriced', 'new_desc_format', 'band_id', 'selling_band_id', 'art_id', 'download_desc_id', 'track_number', 'release_date', 'file_name', 'lyrics', 'album_id', 'encodings_id', 'pending_encodings_id', 'license_type', 'isrc', 'preorder_download', 'streaming', 'id', 'type'])

In [16]:
t_tralbum['current']['track_number']

12

In [17]:
# async def get_track_data(self, url):
# 	async with self.sem:
# 		r = await self.s.get(url)
# 		soup = BeautifulSoup(r.text or "", "lxml")

# 		try:
# 			t_tralbum = json.loads(soup.select_one("[data-tralbum]").get("data-tralbum", "{}")) # type: ignore
# 		except Exception:
# 			log.exception(f"Couldn't fetch data-tralbum from {url}")
# 			return None

# 		# --- art id checks ---
# 		t_art_id = t_tralbum['art_id']
# 		t_track_num = t_tralbum['current']['track_number']

# 		t_art_url = ARTWORK_URL.format(art_id=art_id)
# 		async with self.lock:
# 			# Check for cached urls
# 			if t_art_url in self.seen_art_url:
# 				return None
# 			self.seen_art_url.add(t_art_url)

# 		img = await self.s.get(t_art_url)
# 		img_hash = hashlib.blake2b(img.content).hexdigest()

# 		async with self.lock:
# 			# Check for hash dedupes
# 			if img_hash in self.seen_hash:
# 				return None
# 			self.seen_hash.add(img_hash)


# 		async with self.lock:
# 			# Check for cached art ids
# 			if t_art_id in self.seen_art_id:
# 				return None
# 			self.seen_art_id.add(t_art_id)

# 		return {t_art_id: t_track_num}

In [18]:
	# async def get_track_data(self, url):
	# 	async with self.sem:
	# 		r = await self.s.get(url)
	# 		soup = BeautifulSoup(r.text or "", "lxml")

	# 		try:
	# 			t_tralbum = json.loads(soup.select_one("[data-tralbum]").get("data-tralbum", "{}")) # type: ignore
	# 		except Exception:
	# 			log.exception(f"Couldn't fetch icon_url from {url}")
	# 			return None

	# 		# --- art id checks ---
	# 		t_art_url = ARTWORK_URL.format(art_id=art_id)
	# 		async with self.lock:
	# 			# Check for cached urls
	# 			if t_art_url in self.seen_art_url:
	# 				return None
	# 			self.seen_art_url.add(t_art_url)

	# 		img = await self.s.get(t_art_url)
	# 		img_hash = hashlib.blake2b(img.content).hexdigest()

	# 		async with self.lock:
	# 			# Check for hash dedupes
	# 			if img_hash in self.seen_hash:
	# 				return None
	# 			self.seen_hash.add(img_hash)

	# 		t_art_id = t_art_url.replace("https://f4.bcbits.com/img/a","").split("_", 1)[0]

	# 		return t_art_id

In [19]:
	# async def get_release_data(self, url):
	# 	async with self.sem:
	# 		r = await self.s.get(url)
	# 		soup = BeautifulSoup(r.text or "", "lxml")

	# 		try:
	# 			t_tralbum = json.loads(soup.select_one("[data-tralbum]").get("data-tralbum", "{}"))
	# 		except Exception:
	# 			log.exception(f"Couldn't fetch data-tralbum from {url}")
	# 			return None

	# 		# --- art id checks ---
	# 		t_art_id = str(t_tralbum.get('art_id')).zfill(10)

	# 		async with self.lock:
	# 			# Check for cached art ids
	# 			if t_art_id in self.seen_art_id:
	# 				return None
	# 			self.seen_art_id.add(t_art_id)

	# 		img = await self.s.get(ARTWORK_URL.format(art_id=t_art_id))
	# 		img_hash = hashlib.blake2b(img.content,digest_size=8).hexdigest()

	# 		async with self.lock:
	# 			# Check for hash dedupes
	# 			if img_hash in self.seen_hash:
	# 				return None
	# 			self.seen_hash.add(img_hash)

	# 		return t_art_id, img_hash

### Metadata of one album

### Fetching soup from urls: album page

In [20]:
urls = [
	'https://desertsand.bandcamp.com/album/perli-tal-passat',
	'https://noproblematapes.bandcamp.com/album/--89',
	# 'https://geometriclullaby.bandcamp.com/album/geo-c07'
	]
	
soups = await fetch(urls)

print("Albums fetched:")
print([soup.title.get_text(strip=True) for soup in soups])

Albums fetched:
['Perli tal-Passat | desert sand feels warm at night & days of blue skies | desert sand feels warm at night', 'ලෝක දෙකක් අතර | days of blue skies | NO PROBLEMA TAPES']


In [21]:
import json
album_schema = json.loads(soups[0].select("script[type='application/ld+json']")[0].get_text(strip=True))
tralbum_tag = soups[0].select_one("[data-tralbum]")
tralbum = json.loads(tralbum_tag.get("data-tralbum", "{}"))

[nozero(album_schema['name']), # album name
 nozero(album_schema['byArtist']['name']), # artist name
 album_schema['numTracks'], # number of tracks
 album_schema['keywords'] # tags
]

['Perli tal-Passat',
 'desert sand feels warm at night & days of blue skies',
 14,
 ['Electronic',
  'ambient',
  'chill',
  'experimental electronic',
  'memories',
  'rain',
  'slushwave',
  'smooth',
  'vaporwave',
  'United Kingdom']]

In [22]:
album_schema.keys()

dict_keys(['albumReleaseType', '@id', 'mainEntityOfPage', '@type', 'name', 'dateModified', 'albumRelease', 'byArtist', 'publisher', 'numTracks', 'track', 'image', 'keywords', 'datePublished', 'description', 'creditText', 'copyrightNotice', 'comment', 'sponsor', 'additionalProperty', '@context'])

In [23]:
album_schema['track']['itemListElement']

[{'@type': 'ListItem',
  'position': 1,
  'item': {'@type': 'MusicRecording',
   '@id': 'https://desertsand.bandcamp.com/track/mintix',
   'additionalProperty': [{'@type': 'PropertyValue',
     'name': 'track_id',
     'value': 2597899836},
    {'@type': 'PropertyValue',
     'name': 'license_name',
     'value': 'all_rights_reserved'}],
   'name': 'Mintix',
   'duration': 'P00H09M03S',
   'copyrightNotice': 'All Rights Reserved',
   'mainEntityOfPage': 'https://desertsand.bandcamp.com/track/mintix'}},
 {'@type': 'ListItem',
  'position': 2,
  'item': {'@type': 'MusicRecording',
   '@id': 'https://desertsand.bandcamp.com/track/mog-dija-tal-primula',
   'additionalProperty': [{'@type': 'PropertyValue',
     'name': 'track_id',
     'value': 2030139825},
    {'@type': 'PropertyValue',
     'name': 'license_name',
     'value': 'all_rights_reserved'}],
   'name': 'Mogħdija tal-Primula',
   'duration': 'P00H12M31S',
   'copyrightNotice': 'All Rights Reserved',
   'mainEntityOfPage': 'https

In [24]:
album_schema['description']

'Mhux dawk kollha li jduru jintilfu...\r\n\r\nTracks 1, 4, 5, 7, 9, 10, 14 by\r\ndesert sand feels warm at night\r\n\r\nTracks 2, 3, 6, 8, 11, 12, 13 by\r\ndays of blue skies'

In [25]:
album_schema['creditText']

'days of blue skies:\r\nhttps://daysofblue.bandcamp.com/\r\n\r\nCassette:\r\n\r\nhttps://ephemeraleternal.bandcamp.com/album/perli-tal-passat\r\nhttps://ephemeraleternal.bandcamp.com/album/perli-tal-passat\r\nhttps://ephemeraleternal.bandcamp.com/album/perli-tal-passat'

#### Get alternative album urls from labels in description/credits

In [26]:
import re

ALT_ALBUM_URL = re.compile(r"https://[a-zA-Z0-9-]+\.bandcamp\.com/album/\S+")

def extract_alt_album_urls(album_schema):
	text = (
		(album_schema.get("creditText") or "") + " " +
		(album_schema.get("description") or "")
	)
	links = ALT_ALBUM_URL.findall(text)
	# dedupe
	seen = set()
	result = []
	for link in links:
		if link not in seen:
			seen.add(link)
			result.append(link)

	return result

In [27]:
extract_alt_album_urls(album_schema)

['https://ephemeraleternal.bandcamp.com/album/perli-tal-passat']

In [28]:
tralbum.keys()

dict_keys(['for the curious', 'current', 'preorder_count', 'hasAudio', 'art_id', 'packages', 'defaultPrice', 'freeDownloadPage', 'FREE', 'PAID', 'artist', 'item_type', 'id', 'last_subscription_item', 'has_discounts', 'is_bonus', 'play_cap_data', 'is_purchased', 'items_purchased', 'is_private_stream', 'is_band_member', 'licensed_version_ids', 'package_associated_license_id', 'has_video', 'tralbum_subscriber_only', 'featured_track_id', 'initial_track_num', 'is_preorder', 'album_is_preorder', 'album_release_date', 'trackinfo', 'playing_from', 'url', 'use_expando_lyrics'])

In [29]:
tralbum['id']

1963114278

In [30]:
tralbum['trackinfo']

[{'id': 2597899836,
  'track_id': 2597899836,
  'file': {'mp3-128': 'https://t4.bcbits.com/stream/4218c82fa1717e3fabc1efd6da39f5d3/mp3-128/2597899836?p=0&ts=1781581948&t=466d8c0185d789fd1e1573ec91711baf991c40ea&token=1781581948_29eeb6bc056c7a922bee186785d7bbba7a3c4d5b'},
  'artist': None,
  'title': 'Mintix',
  'encodings_id': 1997498610,
  'license_type': 1,
  'private': None,
  'track_num': 1,
  'album_preorder': False,
  'unreleased_track': False,
  'title_link': '/track/mintix',
  'has_lyrics': False,
  'has_info': False,
  'streaming': 1,
  'is_downloadable': True,
  'has_free_download': None,
  'free_album_download': False,
  'duration': 543.115,
  'lyrics': None,
  'sizeof_lyrics': 0,
  'is_draft': False,
  'video_source_type': None,
  'video_source_id': None,
  'video_mobile_url': None,
  'video_poster_url': None,
  'video_id': None,
  'video_caption': None,
  'video_featured': None,
  'alt_link': None,
  'encoding_error': None,
  'encoding_pending': None,
  'play_count': None,

In [31]:
tralbum['trackinfo'][0]['title_link']

'/track/mintix'

In [32]:
# Get all track urls
import isodate
import pandas as pd
df = pd.DataFrame([
	{
		"url": t['item']['@id']
	}
	for t in album_schema["track"]["itemListElement"]
])
print(df)

                                                  url
0        https://desertsand.bandcamp.com/track/mintix
1   https://desertsand.bandcamp.com/track/mog-dija...
2     https://desertsand.bandcamp.com/track/frammenti
3   https://desertsand.bandcamp.com/track/jiel-tal...
4   https://desertsand.bandcamp.com/track/modulazz...
5      https://desertsand.bandcamp.com/track/tfakkira
6   https://desertsand.bandcamp.com/track/meta-kie...
7    https://desertsand.bandcamp.com/track/solipsi-mu
8   https://desertsand.bandcamp.com/track/qatt-ma-...
9   https://desertsand.bandcamp.com/track/g-ajnejn...
10  https://desertsand.bandcamp.com/track/xorta-mi...
11  https://desertsand.bandcamp.com/track/kisi-tal...
12  https://desertsand.bandcamp.com/track/effu-jon...
13  https://desertsand.bandcamp.com/track/ritmu-ta...


#### Get unique track arts
This is how bandcamp art ID works:
1. Bandcamp generates a unique art_ID for each time you upload a track art.
2. The rest uses art_ID from the album art.

We can drop duplicates based on image hash, with some cases we have to delete manually (dobs album).

In [33]:
import re
import hashlib

async def get_art_id(url):
	session = BrowserSession()
	r = await session.get(url)
	soup = BeautifulSoup(r.text or "", "lxml")

	icon_url = soup.select_one('link[rel="shortcut icon"]')['href']

	img = await session.get(icon_url)
	img_hash = hashlib.sha256(img.content).hexdigest()

	art_id = re.search(r'a(\d+)_', icon_url).group(1)

	return art_id, img_hash


results = await asyncio.gather(
    *(get_art_id(url) for url in df["url"])
)

art_df = pd.DataFrame(
    results,
    columns=["art_id", "img_hash"]
)

print(art_df)

TLSClientException: failed to do request: Get "https://desertsand.bandcamp.com/track/tfakkira": dial tcp: lookup desertsand.bandcamp.com: no such host

In [ ]:
print(f"Total unique art IDs: {art_df['art_id'].nunique()}, Unique Image Hashes: {art_df['img_hash'].nunique()}")

Total unique art IDs: 1, Unique Image Hashes: 1


#### Get all dates of the album

In [ ]:
tralbum = json.loads(tralbum_tag["data-tralbum"])['current']
tralbum2 = json.loads(tralbum_tag["data-tralbum"])['trackinfo']

In [ ]:
[nozero(tralbum['title']), # album name
 tralbum['art_id'], # album image id: 2569838638 in https://f4.bcbits.com/img/a2569838638_10.jpg
 tralbum['new_date'], # date first created draft
 tralbum['mod_date'], # date last modified
 tralbum['publish_date'], # publication date
 tralbum['release_date'] # release date
 ]

['Perli tal-Passat',
 3561606714,
 '16 Dec 2024 00:19:06 GMT',
 '15 Jul 2025 14:55:17 GMT',
 '16 Dec 2024 17:49:51 GMT',
 '16 Dec 2024 00:00:00 GMT']

In [ ]:
# Get all track durations
import pandas as pd
track_df = pd.DataFrame([
	{
		"position": t['track_num'],
		"duration": t['duration'],
	}
	for t in tralbum2
])
print(track_df)

    position  duration
0          1   543.115
1          2   751.118
2          3   600.377
3          4   435.155
4          5   644.972
5          6   636.311
6          7   549.659
7          8   645.083
8          9   726.842
9         10   631.239
10        11   535.385
11        12   438.447
12        13   818.684
13        14   880.626


#### Adding up total runtime of the album:

In [ ]:
from datetime import timedelta
print(timedelta(seconds=int(track_df["duration"].sum())))

2:27:17
